# Documentation-led tabular profiling

Profile CSV and XLSX structures without exposing questionnaire or medication responses.

Documentation authority: [Zenodo record](https://zenodo.org/records/8089820), [Scientific Data descriptor](https://www.nature.com/articles/s41597-023-02445-z), and the publisher documentation retained through `data/raw/`. Unknown semantics always force preserve-only behavior.

## Paths and safety guards

The project is a sibling of the untouched `BCI Database/`. `data/raw/` is a read-only relative symlink to that source, while `data/processed/` is the derived APFS clone.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {"notebooks", "data"}:
    PROJECT_ROOT = PROJECT_ROOT.parent
required_directories = ("data", "models", "notebooks", "results")
if not all((PROJECT_ROOT / name).is_dir() for name in required_directories):
    raise RuntimeError("Run this notebook from bci_cleaning/ or its data/ or notebooks/ directory")

RAW_PATH = PROJECT_ROOT / "data" / "raw"
CLEANED_PATH = PROJECT_ROOT / "data" / "processed"
REPORTS_PATH = PROJECT_ROOT / "results" / "reports"
LOGS_PATH = PROJECT_ROOT / "results" / "logs"
EXPECTED_RAW = PROJECT_ROOT.parent / "BCI Database"

if not RAW_PATH.is_symlink() or RAW_PATH.resolve(strict=True) != EXPECTED_RAW.resolve(strict=True):
    raise RuntimeError("data/raw must remain a relative symlink to the sibling BCI Database directory")

os.chdir(PROJECT_ROOT)
support_path = str(PROJECT_ROOT / "support")
if support_path not in sys.path:
    sys.path.insert(0, support_path)


## Phase implementation

In [2]:
"""Profile every tabular file without modifying or exposing sensitive values."""

from __future__ import annotations

import csv
import io
import json
import sys
from collections import Counter
from pathlib import Path
from typing import Any

from bci_core import (
    ARTICLE_URL,
    Issue,
    atomic_write_csv,
    atomic_write_text,
    common_parser,
    detect_text_encoding,
    iter_files,
    log_event,
    merge_issues,
    new_run_id,
    parse_csv_text,
    resolve_raw,
    sniff_delimiter,
)


PROFILE_FIELDS = [
    "relative_path",
    "container",
    "table",
    "encoding",
    "delimiter",
    "row_count",
    "nonempty_row_count",
    "min_columns",
    "max_columns",
    "missing_cell_count",
    "exact_duplicate_nonempty_rows",
    "formula_cells",
    "data_type_counts",
    "malformed_rows",
    "notes",
]


def _csv_profile(raw: Path, path: Path) -> tuple[dict[str, Any], list[Issue]]:
    relative = path.relative_to(raw).as_posix()
    payload = path.read_bytes()
    issues: list[Issue] = []
    encoding = detect_text_encoding(payload)
    text = payload.decode(encoding)
    delimiter = sniff_delimiter(text)
    rows = parse_csv_text(text, delimiter)
    nonempty = [row for row in rows if any(cell != "" for cell in row)]
    widths = [len(row) for row in nonempty]
    row_counts = Counter(tuple(row) for row in nonempty)
    duplicates = sum(count - 1 for count in row_counts.values() if count > 1)
    max_columns = max(widths, default=0)
    missing = sum(max_columns - len(row) + sum(cell == "" for cell in row) for row in nonempty)
    malformed = sum(width != max_columns for width in widths)
    notes: list[str] = []

    if path.name == "Perfomances.csv":
        notes.append("multi-section participant table; repeated headers and separator rows are structural")
        if encoding != "cp1252":
            issues.append(
                Issue(
                    "warning", "profile", relative, "", "performance_csv_encoding", encoding,
                    "Publisher CSV is expected to decode losslessly as CP1252", ARTICLE_URL,
                    "Preserve and report; clean only after lossless equivalence checks.",
                )
            )
    elif "frequency-band-selected" in path.name:
        notes.append("MDFB algorithm output; numeric values are scientific data")
        if b"\r\r\n" in payload:
            notes.append("CRCRLF line-ending artifact detected")
        else:
            issues.append(
                Issue(
                    "warning", "profile", relative, "", "frequency_csv_line_endings", "CRCRLF absent",
                    "Only observed CRCRLF artifacts are allowlisted for normalization", ARTICLE_URL,
                    "Preserve the file unchanged unless its format is documented.",
                )
            )

    if malformed:
        issues.append(
            Issue(
                "error", "profile", relative, "", "csv_column_consistency", str(malformed),
                "All nonempty records in a documented table section must have consistent width",
                ARTICLE_URL, "Inspect the parser and documentation; do not repair by inference.",
            )
        )
    return (
        {
            "relative_path": relative,
            "container": "csv",
            "table": path.name,
            "encoding": encoding,
            "delimiter": repr(delimiter),
            "row_count": len(rows),
            "nonempty_row_count": len(nonempty),
            "min_columns": min(widths, default=0),
            "max_columns": max_columns,
            "missing_cell_count": missing,
            "exact_duplicate_nonempty_rows": duplicates,
            "formula_cells": 0,
            "data_type_counts": json.dumps({"text": sum(len(row) for row in nonempty)}, sort_keys=True),
            "malformed_rows": malformed,
            "notes": "; ".join(notes),
        },
        issues,
    )


def _xlsx_profiles(raw: Path, path: Path) -> tuple[list[dict[str, Any]], list[Issue]]:
    relative = path.relative_to(raw).as_posix()
    issues: list[Issue] = []
    try:
        import openpyxl
    except ImportError as exc:
        raise RuntimeError("openpyxl is required to profile XLSX workbooks") from exc

    workbook = openpyxl.load_workbook(path, read_only=False, data_only=False)
    profiles: list[dict[str, Any]] = []
    try:
        for sheet in workbook.worksheets:
            rows = list(sheet.iter_rows(min_row=1, max_row=sheet.max_row, min_col=1, max_col=sheet.max_column))
            nonempty_rows = [row for row in rows if any(cell.value is not None for cell in row)]
            row_values = [tuple(cell.value for cell in row) for row in nonempty_rows]
            duplicates = sum(count - 1 for count in Counter(row_values).values() if count > 1)
            formula_cells = sum(
                isinstance(cell.value, str) and cell.value.startswith("=")
                for row in rows
                for cell in row
            )
            types = Counter(cell.data_type for row in rows for cell in row if cell.value is not None)
            missing = sum(cell.value is None for row in rows for cell in row)
            profiles.append(
                {
                    "relative_path": relative,
                    "container": "xlsx",
                    "table": sheet.title,
                    "encoding": "OOXML",
                    "delimiter": "",
                    "row_count": sheet.max_row,
                    "nonempty_row_count": len(nonempty_rows),
                    "min_columns": sheet.max_column,
                    "max_columns": sheet.max_column,
                    "missing_cell_count": missing,
                    "exact_duplicate_nonempty_rows": duplicates,
                    "formula_cells": formula_cells,
                    "data_type_counts": json.dumps(dict(sorted(types.items())), sort_keys=True),
                    "malformed_rows": 0,
                    "notes": "Workbook is canonical; formulas, layout, and repeated section headers are preserved",
                }
            )
    finally:
        workbook.close()
    return profiles, issues


def _documentation_summary() -> str:
    return f"""# Documentation Summary

## Sources consulted

- Zenodo dataset record: https://zenodo.org/records/8089820
- Scientific Data descriptor: {ARTICLE_URL}
- Local English and French experiment instructions and checklists
- Local English and French questionnaire documents
- Local Mental Rotation questionnaire
- OpenViBE scenarios, scripts, channel list, and participant notes

## Documented structure

- 87 anonymized participants: A1-A60, B61-B81, and C82-C87.
- Each participant attended one session.
- A complete participant has two baseline GDF files, acquisition runs R1-R2, and online runs R3-R6.
- Each MI run contains 40 trials: 20 left-hand and 20 right-hand motor-imagery trials.
- Signals contain 27 EEG, 3 EOG, and 2 EMG channels sampled at 512 Hz.
- The workbook is the documented source for performance, demographic, questionnaire, personality, and cognitive-profile fields.

## Documented exceptions and constraints

- A1 acquisition runs were reconstructed from a concatenated recording and lack end-of-trial and end-of-run triggers.
- A59 did not complete R5 or R6; associated EEG and filters are absent.
- A9 and A11 used only R1 for frequency-band selection, although both acquisition recordings remain present.
- Thirteen Dataset B participants have documented questionnaire losses.
- C83 has documented ILS and 16PF5 losses.
- Noisy channels/trials and experimenter comments are intentionally published and must not be removed.
- Configuration-name variants and any undocumented missing assets are preserved and reported, never inferred.

## Cleaning authorization

Only line-ending/encoding changes that pass parsed-cell equivalence are authorized. GDF, XLSX, XML, configuration, questionnaire, instruction, timestamp, identifier, and scientific numeric content remain unchanged.
"""


In [3]:
def main() -> int:
    parser = common_parser(__doc__ or "Profile tabular files")
    args = parser.parse_args()
    raw = resolve_raw(args.raw)
    args.reports.mkdir(parents=True, exist_ok=True)
    args.logs.mkdir(parents=True, exist_ok=True)
    run_id = new_run_id()
    log_event(args.logs, run_id, "profile", "start", raw=str(raw))
    profiles: list[dict[str, Any]] = []
    issues: list[Issue] = []

    for path in iter_files(raw):
        suffix = path.suffix.lower()
        if suffix == ".csv":
            profile, file_issues = _csv_profile(raw, path)
            profiles.append(profile)
            issues.extend(file_issues)
        elif suffix == ".xlsx" and not path.name.startswith("~$"):
            workbook_profiles, file_issues = _xlsx_profiles(raw, path)
            profiles.extend(workbook_profiles)
            issues.extend(file_issues)

    atomic_write_csv(args.reports / "tabular_profile.csv", profiles, PROFILE_FIELDS)
    atomic_write_text(args.reports / "documentation_summary.md", _documentation_summary())
    merge_issues(args.reports / "validation_issues.csv", issues, {"profile"})
    log_event(
        args.logs, run_id, "profile", "complete", tables=len(profiles), issue_count=len(issues)
    )
    print(f"Profile complete: {len(profiles)} tabular objects")
    return 0 if not any(issue.severity in {"fatal", "error"} for issue in issues) else 2


## Execute phase

In [4]:
_arguments = [
    "02_profile.ipynb",
    "--raw", str(RAW_PATH),
    "--cleaned", str(CLEANED_PATH),
    "--reports", str(REPORTS_PATH),
    "--logs", str(LOGS_PATH),
]

_original_argv = sys.argv[:]
try:
    sys.argv = _arguments
    _exit_code = main()
finally:
    sys.argv = _original_argv

if _exit_code != 0:
    raise RuntimeError(f"Phase returned nonzero status {_exit_code}; inspect validation_issues.csv")


Profile complete: 87 tabular objects


## Privacy-safe report check

In [5]:
import csv

report_path = REPORTS_PATH / "tabular_profile.csv"
with report_path.open("r", encoding="utf-8", newline="") as handle:
    report_rows = list(csv.DictReader(handle))

{
    "report": str(report_path.relative_to(PROJECT_ROOT)),
    "rows": len(report_rows),
}


{'report': 'results/reports/tabular_profile.csv', 'rows': 87}